# BTS On-Time Performance — Exploratory Data Audit (pandas)

**Sample:** Reporting-Carrier On-Time Performance, **March 2026** (one monthly download from transtats.bts.gov).

Two passes: (1) an audit of the **raw 110-column download**, and (2) EDA on the **18 columns we keep** after typing/cleaning.

## Part 1 — Raw download audit (all 110 columns)

Read everything as text first, so nothing is coerced or dropped.

In [1]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
CSV = r"data/raw/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_3.csv"
raw = pd.read_csv(CSV, dtype=str, low_memory=False)
raw.shape

(612102, 110)


### Missingness across the raw file
Bucket all 110 columns by their NA fraction.

In [2]:
na_frac = raw.isna().mean()
print('Total columns:', raw.shape[1], '| rows:', raw.shape[0])
print('Fully empty (100% NA):', int((na_frac == 1).sum()))
print('Mostly empty (>90% NA):', int((na_frac > 0.9).sum()))
print('Complete (0% NA):', int((na_frac == 0).sum()))
print('\n15 emptiest columns (NA fraction):')
na_frac.sort_values(ascending=False).head(15).round(3)

Total columns: 110 | rows: 612102
Fully empty (100% NA): 25
Mostly empty (>90% NA): 49
Complete (0% NA): 38

15 emptiest columns (NA fraction):
Div5WheelsOn        1.0
Div5TotalGTime      1.0
Div5LongestGTime    1.0
Div5WheelsOff       1.0
Div5TailNum         1.0
Div5Airport         1.0
Unnamed: 109        1.0
Div4TailNum         1.0
Div4WheelsOff       1.0
Div4LongestGTime    1.0
Div4TotalGTime      1.0
Div4WheelsOn        1.0
Div4AirportSeqID    1.0
Div4AirportID       1.0
Div5AirportID       1.0
dtype: float64


### The fully-empty columns
Note the nameless trailing column (an artifact of every row ending in a comma) and the `Div3`–`Div5` diverted-flight blocks.

In [3]:
list(na_frac[na_frac == 1].index)

['Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn', 'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum', 'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn', 'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum', 'Unnamed: 109']


### Redundancy: derivable date parts
`Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek` are all recomputable from `FlightDate`.

In [4]:
raw[['Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate']].head(3)

   Year Quarter Month DayofMonth DayOfWeek  FlightDate
0  2026       1     3         13         5  2026-03-13
1  2026       1     3         14         6  2026-03-14
2  2026       1     3         14         6  2026-03-14


### Redundancy: multiple airport identifier systems
The origin airport is encoded four different ways; we keep only the human-readable code.

In [5]:
raw[['OriginAirportID','OriginCityMarketID','Origin','OriginCityName']].nunique()

OriginAirportID       342
OriginCityMarketID    319
Origin                342
OriginCityName        336
dtype: int64


## Part 2 — EDA on the 18 columns we keep
Re-read with explicit types (only needed columns), then derive `dep_hour`, `on_time`, and flags.

In [6]:
cols = ['FlightDate','Reporting_Airline','Flight_Number_Reporting_Airline','Origin','Dest',
        'CRSDepTime','DepDelay','ArrDelay','Cancelled','CancellationCode','Diverted',
        'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
df = pd.read_csv(CSV, usecols=cols, parse_dates=['FlightDate'],
                 dtype={'Reporting_Airline':str,'Flight_Number_Reporting_Airline':str,
                        'Origin':str,'Dest':str,'CRSDepTime':str,'CancellationCode':str})
df = df.rename(columns={'FlightDate':'flight_date','Reporting_Airline':'carrier',
    'Flight_Number_Reporting_Airline':'flight_number','Origin':'origin','Dest':'dest',
    'CRSDepTime':'crs_dep_time','DepDelay':'dep_delay','ArrDelay':'arr_delay',
    'Cancelled':'cancelled','CancellationCode':'cancellation_code','Diverted':'diverted',
    'CarrierDelay':'carrier_delay','WeatherDelay':'weather_delay','NASDelay':'nas_delay',
    'SecurityDelay':'security_delay','LateAircraftDelay':'late_aircraft_delay'})
df['dep_hour'] = df['crs_dep_time'].str[:2].astype(int)
df['cancelled'] = df['cancelled'].astype(bool)
df['diverted'] = df['diverted'].astype(bool)
df['on_time'] = (df['arr_delay'] < 15) & (~df['cancelled']) & (~df['diverted'])
df.shape

(612102, 18)


### Missingness in the kept columns
Every NA is *structural*: delays NA ⇒ cancelled/diverted; cause columns NA ⇒ flight wasn't 15+ late.

In [7]:
(df.isna().mean() * 100).round(2)

flight_date             0.00
carrier                 0.00
flight_number           0.00
origin                  0.00
dest                    0.00
crs_dep_time            0.00
dep_delay               2.80
arr_delay               3.24
cancelled               0.00
cancellation_code      97.10
diverted                0.00
carrier_delay          76.47
weather_delay          76.47
nas_delay              76.47
security_delay         76.47
late_aircraft_delay    76.47
dep_hour                0.00
on_time                 0.00
dtype: float64


### Numeric distributions — delays and departure hour
Negative = early.

In [8]:
df[['dep_delay','arr_delay','dep_hour']].describe().round(2)

       dep_delay  arr_delay   dep_hour
count  594946.00  592256.00  612102.00
mean       17.04      10.55      13.04
std        64.66      67.09       4.96
min       -56.00     -90.00       0.00
25%        -6.00     -17.00       9.00
50%        -2.00      -6.00      13.00
75%        15.00      14.00      17.00
max      2888.00    2862.00      23.00


### The five cause-of-delay columns
Minutes, not flags — populated only for flights 15+ min late (hence mostly NA). Among delayed flights, which cause contributes the most total minutes?

In [9]:
causes = ['carrier_delay','weather_delay','nas_delay','security_delay','late_aircraft_delay']
print('% missing per cause column:')
print((df[causes].isna().mean() * 100).round(1))
delayed = df[df['arr_delay'] >= 15]
share = delayed[causes].sum()
print('\nShare of total delay minutes by cause (delayed flights only):')
(100 * share / share.sum()).round(1).sort_values(ascending=False)

% missing per cause column:
carrier_delay          76.5
weather_delay          76.5
nas_delay              76.5
security_delay         76.5
late_aircraft_delay    76.5
dtype: float64

Share of total delay minutes by cause (delayed flights only):
late_aircraft_delay    39.3
carrier_delay          33.5
nas_delay              21.0
weather_delay           5.9
security_delay          0.2
dtype: float64


### Categorical — carrier share (top 15)

In [10]:
vc = df['carrier'].value_counts().head(15)
pd.DataFrame({'n': vc, 'pct': (100 * vc / len(df)).round(1)})

              n   pct
carrier              
WN       122298  20.0
DL        86890  14.2
AA        84647  13.8
OO        73391  12.0
UA        70757  11.6
YX        31734   5.2
AS        27793   4.5
MQ        26721   4.4
B6        21340   3.5
OH        20938   3.4
F9        20119   3.3
NK        12952   2.1
G4        12522   2.0


### Categorical — operational flags & cancellation reasons
Cancellation codes: A=carrier, B=weather, C=NAS (air-system), D=security.

In [11]:
print('on-time %:', round(100 * df['on_time'].mean(), 1))
print('cancelled %:', round(100 * df['cancelled'].mean(), 2))
print('diverted %:', round(100 * df['diverted'].mean(), 2))
cc = df.loc[df['cancelled'], 'cancellation_code'].value_counts()
pd.DataFrame({'n': cc, 'pct': (100 * cc / cc.sum()).round(1)})

on-time %: 73.2
cancelled %: 2.9
diverted %: 0.34
                       n   pct
cancellation_code             
B                  10027  56.4
A                   4850  27.3
C                   2873  16.2
D                     16   0.1


### Categorical — departure hour & busiest origins

In [12]:
dh = df['dep_hour'].value_counts().sort_index()
print('Flights by scheduled departure hour:')
print(pd.DataFrame({'n': dh, 'pct': (100 * dh / len(df)).round(1)}))
print('\nTop 10 origin airports:')
oc = df['origin'].value_counts().head(10)
pd.DataFrame({'n': oc, 'pct': (100 * oc / len(df)).round(1)})

Flights by scheduled departure hour:
              n  pct
dep_hour            
0           840  0.1
1           442  0.1
2           120  0.0
3            66  0.0
4            41  0.0
5         18857  3.1
6         42096  6.9
7         45189  7.4
8         40946  6.7
9         33111  5.4
10        38336  6.3
11        36260  5.9
12        36331  5.9
13        35315  5.8
14        34555  5.6
15        33524  5.5
16        35729  5.8
17        37351  6.1
18        36605  6.0
19        35074  5.7
20        28079  4.6
21        22874  3.7
22        15000  2.5
23         5361  0.9

Top 10 origin airports:
            n  pct
origin            
ORD     31373  5.1
DEN     27089  4.4
ATL     26669  4.4
DFW     26008  4.2
PHX     19277  3.1
LAX     16088  2.6
MCO     15497  2.5
CLT     15278  2.5
LAS     15169  2.5
SEA     12830  2.1
